# Notebook 04 — BERT / DistilBERT Classification

**Purpose**: Fine-tune transformer models for sarcasm classification.
- Task A: Binary (sarcastic vs non-sarcastic)
- Task B: Sarcasm type (6-class, sarcastic only)

**Models**:
- `distilbert-base-uncased` (primary, fast)
- `bert-base-uncased` (optional, if compute allows)

**Prerequisite**: Run `01_data_preparation.ipynb` first.

**Outputs** in `outputs/bert/distilbert_binary/` and `outputs/bert/distilbert_type/`

In [ ]:
import json
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Robust project-root finder ─────────────────────────────────────────────
def _find_project_root() -> Path:
    """Walk up from cwd until we find outputs/splits or another project marker."""
    markers = [
        Path("outputs") / "splits",
        Path("outputs") / "datasets",
        Path("data") / "processed",
        Path("notebooks"),
    ]
    for root in [Path.cwd()] + list(Path.cwd().parents):
        for marker in markers:
            if (root / marker).exists():
                return root
    return Path.cwd().parent  # fallback

ROOT     = _find_project_root()
SPLITS   = ROOT / "outputs" / "splits"
BERT_OUT = ROOT / "outputs" / "bert"
BERT_OUT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")


## Configuration

In [ ]:
@dataclass
class TrainConfig:
    model_name   : str   = "distilbert-base-uncased"
    max_length   : int   = 128
    batch_size   : int   = 32
    lr           : float = 2e-5
    weight_decay : float = 0.01
    warmup_ratio : float = 0.1
    epochs       : int   = 10
    patience     : int   = 3       # early stopping patience
    seed         : int   = SEED
    task         : str   = "binary"  # 'binary' or 'type'
    use_class_weights: bool = False

# â”€â”€ Config for binary task â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
CFG_BINARY = TrainConfig(
    model_name="distilbert-base-uncased",
    task="binary",
    batch_size=32,
    lr=2e-5,
    use_class_weights=False,
)

# â”€â”€ Config for type task (use class weights for imbalance) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
CFG_TYPE = TrainConfig(
    model_name="distilbert-base-uncased",
    task="type",
    batch_size=32,
    lr=2e-5,
    use_class_weights=True,
)

print("Binary config:", CFG_BINARY)
print("Type config:  ", CFG_TYPE)

## Dataset Class

In [ ]:
class HeadlineDataset(Dataset):
    """PyTorch Dataset for headline classification."""

    def __init__(
        self,
        texts: list[str],
        labels: list[int],
        tokenizer,
        max_length: int = 128,
    ):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids"      : encoding["input_ids"].squeeze(0),
            "attention_mask" : encoding["attention_mask"].squeeze(0),
            "labels"         : torch.tensor(self.labels[idx], dtype=torch.long),
        }

## Training and Evaluation Functions

In [ ]:
def compute_class_weights(y_train: list[int], num_classes: int) -> torch.Tensor:
    """Compute inverse-frequency class weights."""
    counts = np.bincount(y_train, minlength=num_classes).astype(float)
    weights = len(y_train) / (num_classes * counts)
    weights = np.clip(weights, 0, 10)  # cap extreme weights
    return torch.tensor(weights, dtype=torch.float)


def train_epoch(
    model, loader: DataLoader, optimizer, scheduler, criterion, device
) -> float:
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits

        loss = criterion(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def eval_epoch(
    model, loader: DataLoader, criterion, device, label_names: list[str]
) -> tuple[float, dict, list]:
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits
        loss    = criterion(logits, labels)

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    metrics = {
        "loss"            : avg_loss,
        "accuracy"        : accuracy_score(all_labels, all_preds),
        "f1_macro"        : f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "f1_weighted"     : f1_score(all_labels, all_preds, average="weighted", zero_division=0),
        "precision_macro" : precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "recall_macro"    : recall_score(all_labels, all_preds, average="macro", zero_division=0),
    }
    return avg_loss, metrics, all_preds


def save_confusion_matrix(y_true, y_pred, label_names, out_path, title=""):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(label_names)))
    fig, ax = plt.subplots(figsize=(max(6, len(label_names)), max(5, len(label_names))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

## Full Training Function

In [ ]:
def train_bert(
    cfg: TrainConfig,
    X_train: list[str], y_train: list[int],
    X_val:   list[str], y_val:   list[int],
    X_test:  list[str], y_test:  list[int],
    label_names: list[str],
    out_dir: Path,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> dict:
    """Full training loop with early stopping. Returns test metrics dict."""
    out_dir.mkdir(parents=True, exist_ok=True)
    num_labels = len(label_names)

    # Save config
    with open(out_dir / "config.json", "w") as f:
        json.dump(cfg.__dict__, f, indent=2)
    print(f"Config saved to {out_dir / 'config.json'}")

    # Tokenizer
    print(f"Loading tokenizer: {cfg.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

    # Datasets
    train_ds = HeadlineDataset(X_train, y_train, tokenizer, cfg.max_length)
    val_ds   = HeadlineDataset(X_val,   y_val,   tokenizer, cfg.max_length)
    test_ds  = HeadlineDataset(X_test,  y_test,  tokenizer, cfg.max_length)

    g = torch.Generator(); g.manual_seed(cfg.seed)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  generator=g)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False)

    # Model
    print(f"Loading model: {cfg.model_name} ({num_labels} labels)")
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name, num_labels=num_labels
    ).to(DEVICE)

    # Loss (with optional class weights)
    if cfg.use_class_weights:
        cw = compute_class_weights(y_train, num_labels).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=cw)
        print(f"Class weights: {cw.cpu().numpy().round(3)}")
    else:
        criterion = nn.CrossEntropyLoss()

    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
    )
    total_steps   = len(train_loader) * cfg.epochs
    warmup_steps  = int(total_steps * cfg.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    # â”€â”€ Training loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    best_val_f1   = -1.0
    patience_cnt  = 0
    best_epoch    = 0
    train_log     = []

    for epoch in range(1, cfg.epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE)
        _, val_metrics, _ = eval_epoch(model, val_loader, criterion, DEVICE, label_names)

        val_f1 = val_metrics["f1_macro"]
        log_row = {"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}}
        train_log.append(log_row)

        print(
            f"Epoch {epoch:2d}/{cfg.epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_macro_f1={val_f1:.4f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch  = epoch
            patience_cnt = 0
            # Save best checkpoint
            ckpt_dir = out_dir / "best_checkpoint"
            model.save_pretrained(ckpt_dir)
            tokenizer.save_pretrained(ckpt_dir)
            print(f"  â˜… New best val macro-F1={val_f1:.4f} â€” checkpoint saved")
        else:
            patience_cnt += 1
            if patience_cnt >= cfg.patience:
                print(f"  Early stopping at epoch {epoch} (patience={cfg.patience})")
                break

    # Save training log
    log_df = pd.DataFrame(train_log)
    log_df.to_csv(out_dir / "training_log.csv", index=False)

    # â”€â”€ Plot training curves â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(log_df["epoch"], log_df["train_loss"],  label="Train loss", marker="o")
    axes[0].plot(log_df["epoch"], log_df["val_loss"],    label="Val loss",   marker="s")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{cfg.model_name} â€” Loss Curves ({cfg.task})")
    axes[0].legend(); axes[0].axvline(best_epoch, color="gray", linestyle="--", alpha=0.5)

    axes[1].plot(log_df["epoch"], log_df["val_f1_macro"], label="Val macro-F1", marker="o", color="steelblue")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro-F1")
    axes[1].set_title(f"{cfg.model_name} â€” Val Macro-F1 ({cfg.task})")
    axes[1].legend(); axes[1].axvline(best_epoch, color="gray", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig(out_dir / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

    # â”€â”€ Reload best checkpoint and evaluate â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print(f"\nLoading best checkpoint (epoch {best_epoch}, val macro-F1={best_val_f1:.4f})")
    best_model = AutoModelForSequenceClassification.from_pretrained(
        out_dir / "best_checkpoint"
    ).to(DEVICE)

    _, val_metrics_best,  val_preds  = eval_epoch(best_model, val_loader,  criterion, DEVICE, label_names)
    _, test_metrics_best, test_preds = eval_epoch(best_model, test_loader, criterion, DEVICE, label_names)

    print("\n=== Val (best checkpoint) ===")
    print(classification_report(y_val,  val_preds,  target_names=label_names, zero_division=0))
    print("\n=== Test (best checkpoint) ===")
    print(classification_report(y_test, test_preds, target_names=label_names, zero_division=0))

    # Confusion matrices
    save_confusion_matrix(y_val,  val_preds,  label_names,
                          out_dir / "confusion_matrix_val.png",
                          f"{cfg.model_name} â€” {cfg.task} (Val)")
    save_confusion_matrix(y_test, test_preds, label_names,
                          out_dir / "confusion_matrix_test.png",
                          f"{cfg.model_name} â€” {cfg.task} (Test)")

    # Save metrics
    results = {
        "model": cfg.model_name, "task": cfg.task,
        "best_epoch": best_epoch, "best_val_f1_macro": best_val_f1,
        "val":  val_metrics_best,
        "test": test_metrics_best,
    }
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(results, f, indent=2)

    # Save predictions
    for split_name, df, y_true_list, y_pred_list in [
        ("val",  val_df,  y_val,  val_preds),
        ("test", test_df, y_test, test_preds),
    ]:
        out_pred = df.copy()
        out_pred["predicted"] = y_pred_list
        if hasattr(label_names, '__getitem__'):
            out_pred["predicted_label"] = [label_names[i] for i in y_pred_list]
        out_pred["correct"] = (np.array(y_true_list) == np.array(y_pred_list)).astype(int)
        out_pred.to_csv(out_dir / f"predictions_{split_name}.csv", index=False)
        print(f"Saved predictions_{split_name}.csv")

    print(f"\n=== DONE: {cfg.model_name} / {cfg.task} ===")
    print(f"  Test Accuracy  : {test_metrics_best['accuracy']:.4f}")
    print(f"  Test Macro-F1  : {test_metrics_best['f1_macro']:.4f}")
    print(f"  Test Weighted-F1: {test_metrics_best['f1_weighted']:.4f}")

    return results

## Load Data

In [ ]:
# â”€â”€ Binary splits â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
train_bin = pd.read_csv(SPLITS / "train_binary.csv")
val_bin   = pd.read_csv(SPLITS / "val_binary.csv")
test_bin  = pd.read_csv(SPLITS / "test_binary.csv")

X_train_bin = train_bin["text"].tolist(); y_train_bin = train_bin["binary_label"].tolist()
X_val_bin   = val_bin["text"].tolist();   y_val_bin   = val_bin["binary_label"].tolist()
X_test_bin  = test_bin["text"].tolist();  y_test_bin  = test_bin["binary_label"].tolist()

label_names_bin = ["non-sarcastic", "sarcastic"]

print(f"Binary â€” Train: {len(X_train_bin):,}  Val: {len(X_val_bin):,}  Test: {len(X_test_bin):,}")

In [ ]:
# â”€â”€ Type splits â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
train_type = pd.read_csv(SPLITS / "train_type.csv")
val_type   = pd.read_csv(SPLITS / "val_type.csv")
test_type  = pd.read_csv(SPLITS / "test_type.csv")

STRATEGY_LABELS = sorted(train_type["type_label"].unique())
label2id = {lab: i for i, lab in enumerate(STRATEGY_LABELS)}
id2label = {i: lab for lab, i in label2id.items()}

def enc(df): return [label2id[l] for l in df["type_label"]]

X_train_type = train_type["text"].tolist(); y_train_type = enc(train_type)
X_val_type   = val_type["text"].tolist();   y_val_type   = enc(val_type)
X_test_type  = test_type["text"].tolist();  y_test_type  = enc(test_type)

print(f"Type   â€” Train: {len(X_train_type):,}  Val: {len(X_val_type):,}  Test: {len(X_test_type):,}")
print(f"Strategies: {STRATEGY_LABELS}")

## Train DistilBERT â€” Binary Task

In [ ]:
results_distilbert_binary = train_bert(
    cfg        = CFG_BINARY,
    X_train    = X_train_bin, y_train = y_train_bin,
    X_val      = X_val_bin,   y_val   = y_val_bin,
    X_test     = X_test_bin,  y_test  = y_test_bin,
    label_names= label_names_bin,
    out_dir    = BERT_OUT / "distilbert_binary",
    val_df     = val_bin,
    test_df    = test_bin,
)

## Train DistilBERT â€” Type Task

In [ ]:
results_distilbert_type = train_bert(
    cfg        = CFG_TYPE,
    X_train    = X_train_type, y_train = y_train_type,
    X_val      = X_val_type,   y_val   = y_val_type,
    X_test     = X_test_type,  y_test  = y_test_type,
    label_names= STRATEGY_LABELS,
    out_dir    = BERT_OUT / "distilbert_type",
    val_df     = val_type,
    test_df    = test_type,
)

## (Optional) BERT-base â€” Binary Task

Run only if compute allows. Comment out if not needed.

In [ ]:
# Uncomment to run BERT-base (requires more GPU memory and time)

# CFG_BERT_BINARY = TrainConfig(
#     model_name="bert-base-uncased",
#     task="binary",
#     batch_size=16,  # smaller batch for bert-base
#     lr=2e-5,
# )

# results_bert_binary = train_bert(
#     cfg         = CFG_BERT_BINARY,
#     X_train     = X_train_bin, y_train = y_train_bin,
#     X_val       = X_val_bin,   y_val   = y_val_bin,
#     X_test      = X_test_bin,  y_test  = y_test_bin,
#     label_names = label_names_bin,
#     out_dir     = BERT_OUT / "bert_base_binary",
#     val_df      = val_bin,
#     test_df     = test_bin,
# )

print("BERT-base is commented out. Uncomment cells above to run.")

## (Optional) BERT-base â€” Type Task

In [ ]:
# CFG_BERT_TYPE = TrainConfig(
#     model_name="bert-base-uncased",
#     task="type",
#     batch_size=16,
#     lr=2e-5,
#     use_class_weights=True,
# )

# results_bert_type = train_bert(
#     cfg         = CFG_BERT_TYPE,
#     X_train     = X_train_type, y_train = y_train_type,
#     X_val       = X_val_type,   y_val   = y_val_type,
#     X_test      = X_test_type,  y_test  = y_test_type,
#     label_names = STRATEGY_LABELS,
#     out_dir     = BERT_OUT / "bert_base_type",
#     val_df      = val_type,
#     test_df     = test_type,
# )

print("BERT-base type task is commented out. Uncomment to run.")

## Summary

In [ ]:
print("====== BERT RESULTS SUMMARY ======")
print()
print("DistilBERT â€” Binary (Test):")
print(f"  Accuracy   : {results_distilbert_binary['test']['accuracy']:.4f}")
print(f"  Macro-F1   : {results_distilbert_binary['test']['f1_macro']:.4f}")
print(f"  Weighted-F1: {results_distilbert_binary['test']['f1_weighted']:.4f}")
print()
print("DistilBERT â€” Type (Test):")
print(f"  Accuracy   : {results_distilbert_type['test']['accuracy']:.4f}")
print(f"  Macro-F1   : {results_distilbert_type['test']['f1_macro']:.4f}")
print(f"  Weighted-F1: {results_distilbert_type['test']['f1_weighted']:.4f}")
print()
print("Artifacts saved in outputs/bert/distilbert_binary/ and outputs/bert/distilbert_type/")